In [1]:
import pandas as pd
import polars as pl
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

PROCESSED_DATA_DIR = os.getenv('PROCESSED_DATA_DIR')
FIGURES_DIR = os.getenv('FIGURES_DIR')

In [2]:
df = pd.read_csv(f'{PROCESSED_DATA_DIR}/processed_nidingen_data.csv')
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 54233 entries, 0 to 54232
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             54233 non-null  datetime64[us]
 1   time             50526 non-null  float64       
 2   record_type      54233 non-null  str           
 3   ring_number      54233 non-null  str           
 4   age_code         54233 non-null  str           
 5   species_code     54233 non-null  str           
 6   ringer           54233 non-null  str           
 7   age              54177 non-null  str           
 8   wing_length      54233 non-null  int64         
 9   weight           17353 non-null  float64       
 10  fat_score        54233 non-null  int64         
 11  muscle_score     54233 non-null  int64         
 12  brood_patch      50693 non-null  float64       
 13  moult_score      52464 non-null  float64       
 14  notes            54233 non-null  str           
 

In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [4]:
def preprocess(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week'] = df['date'].dt.isocalendar().week
    return df

def plot_total_ringings_over_time(df):
    df = preprocess(df)
    daily = df.groupby('date').size().reset_index(name='count')
    fig = px.line(daily, x='date', y='count', title='Total Birds Ringed Per Day')
    fig.update_layout(yaxis_title='Birds Ringed')
    return fig

def plot_species_trends(df, top_n=10):
    df = preprocess(df)
    top_species = df['swedish_name'].value_counts().nlargest(top_n).index
    filtered = df[df['swedish_name'].isin(top_species)]
    weekly = filtered.groupby(['week', 'swedish_name']).size().reset_index(name='count')
    weekly['count'] = weekly['count']/df.year.nunique()  # Normalize by number of years
    fig = px.line(weekly, x='week', y='count', color='swedish_name',
                  facet_col='swedish_name', facet_col_wrap=2,
                  title=f'Top {top_n} Species Average by Week of Year')
    return fig

def plot_species_seasonality(df, species_list):
    df = preprocess(df)
    filtered = df[df['swedish_name'].isin(species_list)]
    fig = px.histogram(filtered, x='day_of_year', color='swedish_name', nbins=80,
                       barmode='overlay', title='Seasonality by Species')
    fig.update_layout(xaxis_title='Day of Year', yaxis_title='Ringing Count')
    return fig

def plot_age_distribution(df, species_list):
    df = preprocess(df)
    filtered = df[df['swedish_name'].isin(species_list)]
    age_dist = filtered.groupby(['swedish_name', 'age_code']).size().reset_index(name='count')
    fig = px.bar(age_dist, x='swedish_name', y='count', color='age_code',
                 title='Age Distribution by Species')
    fig.update_layout(xaxis_title='Species', yaxis_title='Count')
    return fig

def plot_ringer_activity(df, top_n=10):
    df = preprocess(df)
    top_ringers = df['ringer'].value_counts().nlargest(top_n).index
    filtered = df[df['ringer'].isin(top_ringers)]
    ringer_stats = filtered.groupby(['date', 'ringer']).size().reset_index(name='count')
    fig = px.line(ringer_stats, x='date', y='count', color='ringer',
                  title=f'Ringing Activity of Top {top_n} Ringers Over Time')
    return fig

def plot_species_month_heatmap(df, species_list):
    df = preprocess(df)
    filtered = df[df['swedish_name'].isin(species_list)]
    fig = px.density_heatmap(filtered, x='week', y='swedish_name',
                             color_continuous_scale='Viridis',
                             title='Species × Month Ringing Heatmap')
    return fig

def summarize_ringings_per_year(df):
    df = preprocess(df)
    summary = df.groupby('year').size().describe()
    return summary

In [13]:
most_common = df.swedish_name.value_counts().nlargest(10).index.tolist()

fig1 = plot_total_ringings_over_time(df)
fig1.show()
# Save figure
fig1.write_image(f'{FIGURES_DIR}/total_ringings_over_time.svg', width=1200, height=800)

fig2 = plot_species_trends(df)
fig2.show()
fig2.write_image(f'{FIGURES_DIR}/species_trends.svg', width=1200, height=800)
#fig3 = plot_species_seasonality(df, most_common[:10])
#fig3.show()
#fig3.write_image(f'{FIGURES_DIR}/species_seasonality.svg', width=1200, height=800)

fig4 = plot_species_month_heatmap(df, most_common[:10])
fig4.show()
fig4.write_image(f'{FIGURES_DIR}/species_month_heatmap.svg', width=1200, height=800)
summary_stats = summarize_ringings_per_year(df)
print(summary_stats)


count        5.000000
mean     10846.600000
std        826.042553
min       9967.000000
25%      10400.000000
50%      10443.000000
75%      11491.000000
max      11932.000000
dtype: float64


In [12]:
df[df.swedish_name=='silltrut']

,date,time,record_type,ring_number,age_code,species_code,ringer,age,wing_length,weight,fat_score,muscle_score,brood_patch,moult_score,notes,scientific_name,swedish_name,TaxonID
0,2020-03-06,NaN,C,8128051,3,SITRU,TJÄ,00,0,0.0,10,10,0.0,0.0,0,Larus fuscus,silltrut,205660.0
616,2020-03-20,NaN,C,8127212,3,SITRU,TJÄ,00,0,0.0,10,10,0.0,0.0,0,Larus fuscus,silltrut,205660.0
617,2020-03-25,NaN,C,8127181,3,SITRU,LBR,00,0,0.0,10,10,0.0,0.0,0,Larus fuscus,silltrut,205660.0
618,2020-03-30,NaN,C,8123784,3,SITRU,UNO,00,0,0.0,10,10,0.0,0.0,0,Larus fuscus,silltrut,205660.0
619,2020-03-30,NaN,C,8127237,3,SITRU,UNO,00,0,0.0,10,10,0.0,0.0,0,Larus fuscus,silltrut,205660.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48042,2024-07-11,NaN,R,8143370,1,SITRU,TJÄ,00,0,565.0,10,10,0.0,0.0,0,Larus fuscus,silltrut,205660.0
48043,2024-07-11,NaN,R,8143371,1,SITRU,TJÄ,00,0,850.0,10,10,0.0,0.0,0,Larus fuscus,silltrut,205660.0
48044,2024-07-11,NaN,R,8143372,1,SITRU,TJÄ,00,0,710.0,10,10,0.0,0.0,0,Larus fuscus,silltrut,205660.0
48045,2024-07-11,NaN,R,8143373,1,SITRU,TJÄ,00,0,820.0,10,10,0.0,0.0,0,Larus fuscus,silltrut,205660.0


In [7]:

fig2 = plot_species_trends(df)
fig2.show()
fig2.write_image(f'{FIGURES_DIR}/species_trends.svg', width=1200, height=800)
fig3 = plot_species_seasonality(df, most_common[:10])
fig3.show()
fig3.write_image(f'{FIGURES_DIR}/species_seasonality.svg', width=1200, height=800)

fig4 = plot_species_month_heatmap(df, most_common[:10])
fig4.show()
fig4.write_image(f'{FIGURES_DIR}/species_month_heatmap.svg', width=1200, height=800)
summary_stats = summarize_ringings_per_year(df)
print(summary_stats)


KeyError: 'silltrut'